In [1]:
import pandas as pd

## 1. Load prepared lyrics dataset

In [2]:
lyrics_df = pd.read_csv(
    "../data/processed/final/lyrics_prepared.csv"
)

In [3]:
lyrics_df.shape

(540, 17)

In [5]:
lyrics_df["lyrics_status"].value_counts(dropna=False)

lyrics_status
retrieved_lyrics    539
instrumental          1
Name: count, dtype: int64

In [6]:
lyrics_df["language"].value_counts(dropna=False)

language
english       523
polish          9
french          3
unknown         1
indonesian      1
italian         1
chinese         1
NaN             1
Name: count, dtype: int64

## 2. Prepare the English lyrics corpus

In [7]:
english_lyrics = lyrics_df[
    (lyrics_df["lyrics_status"] == "retrieved_lyrics") & (lyrics_df["language"] == "english")
].copy()

In [8]:
english_lyrics.shape

(523, 17)

## 3. Load the NRC Emotion Lexicon

In [9]:
nrc_path = (
    "../data/nrc_lexicon/"
    "NRC-Suite-of-Sentiment-Emotion-Lexicons/"
    "NRC-Sentiment-Emotion-Lexicons/"
    "NRC-Emotion-Lexicon-v0.92/"
    "NRC-Emotion-Lexicon-Wordlevel-v0.92.txt"
)

nrc = pd.read_csv(
    nrc_path,
    sep="\t",
    header=None
)

nrc.head()

,0,1,2
0,aback,anger,0
1,aback,anticipation,0
2,aback,disgust,0
3,aback,fear,0
4,aback,joy,0


In [10]:
nrc.columns = ["word", "category", "association"]

nrc.head()

,word,category,association
0,aback,anger,0
1,aback,anticipation,0
2,aback,disgust,0
3,aback,fear,0
4,aback,joy,0


In [11]:
nrc["category"].unique()

<ArrowStringArray>
[       'anger', 'anticipation',      'disgust',         'fear',
          'joy',     'negative',     'positive',      'sadness',
     'surprise',        'trust']
Length: 10, dtype: str

In [12]:
nrc["category"].value_counts()

category
anger           14154
anticipation    14154
disgust         14154
fear            14154
joy             14154
negative        14154
positive        14154
sadness         14154
surprise        14154
trust           14154
Name: count, dtype: int64

In [14]:
# We don't need all rows with association = 0. We're only interested in words that are truly associated by NRC with particular sentiments and emotions.

nrc_active = nrc[nrc["association"] == 1].copy()

nrc_active.head()

,word,category,association
19,abacus,trust,1
23,abandon,fear,1
25,abandon,negative,1
27,abandon,sadness,1
30,abandoned,anger,1


In [15]:
nrc_active["category"].value_counts()

category
negative        3316
positive        2308
fear            1474
anger           1245
trust           1230
sadness         1187
disgust         1056
anticipation     837
joy              687
surprise         532
Name: count, dtype: int64

In [ ]:
# Check case of words and whitespaces

nrc_active["word"] = (nrc_active["word"].str.lower().str.strip())

nrc_active["word"].duplicated().sum()

np.int64(7419)

In [18]:
# Check duplicates of word-category pairs. There should be none.

nrc_active.duplicated(
    subset=["word", "category"]
).sum()

np.int64(0)

In [20]:
nrc_lookup = (nrc_active.groupby("word")["category"].agg(list).to_dict())

In [21]:
nrc_lookup["abandon"]

['fear', 'negative', 'sadness']

In [22]:
nrc_lookup["joy"]

['joy', 'positive']

## 4. Build track-level emotion profiles

In [23]:
import re

def tokenize_lyrics(text):
    return re.findall(r"[a-z]+", str(text).lower())

In [24]:
test_lyrics = english_lyrics["lyrics_clean"].iloc[0]

tokenize_lyrics(test_lyrics)[:30]

['if',
 'i',
 'had',
 'my',
 'own',
 'world',
 'i',
 'd',
 'build',
 'you',
 'an',
 'empire',
 'if',
 'i',
 'had',
 'my',
 'own',
 'world',
 'i',
 'd',
 'fill',
 'it',
 'with',
 'wealth',
 'and',
 'desire',
 'a',
 'glorious',
 'past',
 'to']

In [25]:
def get_nrc_matches(text):
    tokens = tokenize_lyrics(text)
    
    matched_tokens = []
    matched_categories = []
    
    for token in tokens:
        categories = nrc_lookup.get(token)
        if categories:
            matched_tokens.append(token)
            matched_categories.extend(categories)
            
    return {
        "total_tokens": len(tokens),
        "matched_tokens": len(matched_tokens),
        "matched_categories": matched_categories
    }

In [26]:
test_result = get_nrc_matches(test_lyrics)
test_result

{'total_tokens': 373,
 'matched_tokens': 54,
 'matched_categories': ['positive',
  'trust',
  'joy',
  'positive',
  'trust',
  'positive',
  'trust',
  'joy',
  'positive',
  'anger',
  'anticipation',
  'disgust',
  'fear',
  'negative',
  'sadness',
  'surprise',
  'fear',
  'positive',
  'trust',
  'positive',
  'joy',
  'positive',
  'anger',
  'fear',
  'negative',
  'sadness',
  'positive',
  'joy',
  'positive',
  'anger',
  'fear',
  'negative',
  'sadness',
  'anticipation',
  'fear',
  'anger',
  'negative',
  'anticipation',
  'joy',
  'positive',
  'surprise',
  'trust',
  'fear',
  'trust',
  'anticipation',
  'joy',
  'positive',
  'sadness',
  'trust',
  'trust',
  'positive',
  'trust',
  'joy',
  'positive',
  'trust',
  'anticipation',
  'fear',
  'joy',
  'positive',
  'trust',
  'anticipation',
  'joy',
  'positive',
  'surprise',
  'trust',
  'positive',
  'joy',
  'positive',
  'anger',
  'fear',
  'negative',
  'sadness',
  'positive',
  'joy',
  'positive',
  '

In [27]:
# matched_tokens = 54 means that 54 text tokens were found in the NRC lexicon. Not 54 different words, but 54 instances of words that are in the lexicon. For example, if the word "love" appears 10 times in the lyrics, it will be counted as 10 matched tokens.

## 5. Sentiment and emotional balance

## 6. Aggregate emotion profiles by era

## 7. Aggregate emotion profiles by macro-cluster

## 8. Era × macro-cluster comparison

## 9. Visualizations

## 10. Robustness check

## Conclusions